# 04 — ORESTAR City Council spending profiles (2024)

Candidate-level spending profiles for **Portland City Council** using cleaned
ORESTAR transactions for 2024.

The representation mirrors the fundraising notebooks:

- total spending and expenditure-record count;
- mean, median, minimum, maximum, and standard deviation;
- spending-size bins;
- spending amount share and expenditure-record share by bin.

`expenditure_count` is a count of reported expenditure records, not purchases
or unique vendors.


## 1. Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 190)

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find repository root. "
        "Expected pyproject.toml in the current directory or its parent."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from helpers.paths import (
    CLEAN,
    PROCESSED,
    fundraising_processed_dir,
    orestar_file_audit_path,
    orestar_transactions_path,
    spending_processed_dir,
)

print("ROOT:", ROOT)

YEAR = 2024
CONTEST = "city_council"

TRANSACTIONS_PATH = orestar_transactions_path(
    YEAR,
    CONTEST,
)

AUDIT_PATH = orestar_file_audit_path(
    YEAR,
    CONTEST,
)

CROSSWALK_PATH = (
    PROCESSED
    / "master"
    / f"candidate_source_crosswalk_{YEAR}.csv"
)

OUTPUT_DIR = spending_processed_dir(
    YEAR,
    CONTEST,
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Transactions:", TRANSACTIONS_PATH)
print("Audit:", AUDIT_PATH)
print("Crosswalk exists:", CROSSWALK_PATH.exists())
print("Output:", OUTPUT_DIR)


ROOT: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis
Transactions: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/clean/orestar/2024/city_council/transactions.csv
Audit: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/clean/orestar/2024/city_council/file_audit.csv
Crosswalk exists: True
Output: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/spending/2024/city_council


## 2. Load and validate cleaned City Council ORESTAR

In [2]:
def to_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )

transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    low_memory=False,
)

required = {
    "year",
    "contest_type",
    "office",
    "district",
    "source_file_stem",
    "sub_type",
    "is_reported_expenditure",
    "reported_expenditure_amount",
}

missing = sorted(
    required
    - set(transactions.columns)
)

if missing:
    raise ValueError(
        f"Missing clean ORESTAR columns: {missing}"
    )

contest_values = set(
    transactions["contest_type"]
    .dropna()
    .astype(str)
    .unique()
)

if contest_values != {CONTEST}:
    raise ValueError(
        f"Expected only {CONTEST}; found {sorted(contest_values)}"
    )

print("Rows:", f"{len(transactions):,}")
display(
    transactions["sub_type"]
    .value_counts(dropna=False)
    .to_frame("rows")
)


Rows: 35,542


,rows
sub_type,
Cash Contribution,20410
Cash Expenditure,12384
Personal Expenditure for Reimbursement,1081
In-Kind Contribution,403
Account Payable,366
Return or Refund of Contribution,308
Refunds and Rebates,178
Items Sold at Fair Market Value,93
Lost or Returned Check,88


## 3. Keep positive reported expenditures

In [3]:
transactions["reported_expenditure_amount"] = pd.to_numeric(
    transactions["reported_expenditure_amount"],
    errors="coerce",
)

spending = transactions.loc[
    to_bool(
        transactions["is_reported_expenditure"]
    )
    & transactions["reported_expenditure_amount"].notna()
    & transactions["reported_expenditure_amount"].gt(0)
].copy()

spending["amount"] = (
    spending["reported_expenditure_amount"]
)

print("Expenditure records:", f"{len(spending):,}")
print("Total spending:", f"${spending['amount'].sum():,.2f}")


Expenditure records: 13,489
Total spending: $7,853,887.19


## 4. File audit

Duplicate source exports are surfaced, not silently removed.


In [4]:
if AUDIT_PATH.exists():
    audit = pd.read_csv(
        AUDIT_PATH,
        low_memory=False,
    )

    duplicate_exports = audit.loc[
        to_bool(
            audit["is_exact_duplicate_export"]
        )
    ].copy()

    print(
        "Exact duplicate export rows:",
        len(duplicate_exports),
    )

    if len(duplicate_exports):
        display(
            duplicate_exports[
                [
                    "district",
                    "source_file",
                    "source_file_hash",
                    "same_hash_file_count",
                ]
            ]
        )
else:
    print("No file audit found.")


Exact duplicate export rows: 4


,district,source_file,source_file_hash,same_hash_file_count
28,2,Penson.xls,122c395765679c92a70fe95a4abd4e3847067a305539f0...,2
49,3,Morse.xls,f4d27e719c151b49391886c0b2e1ce9f17e5a8e9b3888e...,2
52,3,Penson.xls,122c395765679c92a70fe95a4abd4e3847067a305539f0...,2
67,4,Morse.xls,f4d27e719c151b49391886c0b2e1ce9f17e5a8e9b3888e...,2


## 5. Candidate identity

In [5]:
def attach_orestar_identity(data: pd.DataFrame, crosswalk_path: Path) -> pd.DataFrame:
    frame = data.copy()

    frame["source_candidate_name"] = (
        frame["source_file_stem"]
        .astype("string")
        .str.strip()
    )

    frame["canonical_candidate"] = pd.NA
    frame["candidate_key"] = pd.NA
    frame["linkage_status"] = "source_only"

    if crosswalk_path.exists():
        crosswalk = pd.read_csv(
            crosswalk_path,
            low_memory=False,
        )

        needed = {
            "source",
            "classification",
            "year",
            "district",
            "source_candidate_name",
            "suggested_candidate",
            "suggested_candidate_key",
        }

        if needed.issubset(crosswalk.columns):
            matched = (
                crosswalk.loc[
                    crosswalk["source"].eq("orestar")
                    & crosswalk["classification"].eq("match"),
                    [
                        "year",
                        "district",
                        "source_candidate_name",
                        "suggested_candidate",
                        "suggested_candidate_key",
                    ],
                ]
                .rename(
                    columns={
                        "suggested_candidate": "_canonical_candidate",
                        "suggested_candidate_key": "_candidate_key",
                    }
                )
                .drop_duplicates()
            )

            frame = frame.merge(
                matched,
                on=[
                    "year",
                    "district",
                    "source_candidate_name",
                ],
                how="left",
                validate="many_to_one",
            )

            frame["canonical_candidate"] = frame["_canonical_candidate"]
            frame["candidate_key"] = frame["_candidate_key"]

            frame["linkage_status"] = np.where(
                frame["candidate_key"].notna(),
                "matched_to_official_candidate",
                "unmatched_source_candidate",
            )

            frame = frame.drop(
                columns=[
                    "_canonical_candidate",
                    "_candidate_key",
                ]
            )

    frame["candidate"] = (
        frame["canonical_candidate"]
        .fillna(frame["source_candidate_name"])
    )

    source_key = (
        frame["year"].astype("Int64").astype(str)
        + "|"
        + frame["district"].astype("Int64").astype(str)
        + "|orestar|"
        + frame["source_candidate_name"]
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    frame["profile_key"] = (
        frame["candidate_key"]
        .fillna(source_key)
    )

    return frame

spending = attach_orestar_identity(
    spending,
    CROSSWALK_PATH,
)

identity_summary = (
    spending[
        [
            "year",
            "district",
            "source_file_stem",
            "candidate",
            "candidate_key",
            "profile_key",
            "linkage_status",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "district",
            "candidate",
        ]
    )
)

display(identity_summary)

display(
    identity_summary["linkage_status"]
    .value_counts(dropna=False)
    .to_frame("candidate_source_rows")
)


,year,district,source_file_stem,candidate,candidate_key,profile_key,linkage_status
0,2024,1,Avalos,Candace Avalos,2024|1|candace avalos,2024|1|candace avalos,matched_to_official_candidate
3650,2024,1,Tern,Cayle Tern,2024|1|cayle tern,2024|1|cayle tern,matched_to_official_candidate
1362,2024,1,Linn,David Linn,2024|1|david linn,2024|1|david linn,matched_to_official_candidate
2039,2024,1,Salazar,Deian Salazar,2024|1|deian salazar,2024|1|deian salazar,matched_to_official_candidate
470,2024,1,Durphy,Durphy,NaN,2024|1|orestar|durphy,unmatched_source_candidate
...,...,...,...,...,...,...,...
10710,2024,4,Clark,Olivia Clark,2024|4|olivia clark,2024|4|olivia clark,matched_to_official_candidate
12607,2024,4,Silkie,Sarah Silkie,2024|4|sarah silkie,2024|4|sarah silkie,matched_to_official_candidate
12212,2024,4,Penkin,Stanley Penkin,2024|4|stanley penkin,2024|4|stanley penkin,matched_to_official_candidate
12040,2024,4,Morse,Tony Morse,2024|4|tony morse,2024|4|tony morse,matched_to_official_candidate


,candidate_source_rows
linkage_status,
matched_to_official_candidate,66
unmatched_source_candidate,10


## 6. Candidate spending summary

In [6]:
PROFILE_KEYS = [
    "year",
    "contest_type",
    "office",
    "district",
    "profile_key",
    "candidate",
]

candidate_summary = (
    spending
    .groupby(
        PROFILE_KEYS,
        as_index=False,
        dropna=False,
    )
    .agg(
        total_spending=("amount", "sum"),
        expenditure_count=("amount", "size"),
        average_expenditure=("amount", "mean"),
        median_expenditure=("amount", "median"),
        min_expenditure=("amount", "min"),
        max_expenditure=("amount", "max"),
        std_expenditure=("amount", "std"),
    )
)

identity_cols = (
    spending[
        [
            "profile_key",
            "source_file_stem",
            "source_candidate_name",
            "canonical_candidate",
            "candidate_key",
            "linkage_status",
        ]
    ]
    .drop_duplicates(
        subset=["profile_key"]
    )
)

candidate_summary = candidate_summary.merge(
    identity_cols,
    on="profile_key",
    how="left",
    validate="one_to_one",
)

candidate_summary = candidate_summary.sort_values(
    [
        "district",
        "total_spending",
    ],
    ascending=[
        True,
        False,
    ],
)

display(candidate_summary)


,year,contest_type,office,district,profile_key,candidate,total_spending,expenditure_count,average_expenditure,median_expenditure,min_expenditure,max_expenditure,std_expenditure,source_file_stem,source_candidate_name,canonical_candidate,candidate_key,linkage_status
5,2024,city_council,Portland City Council,1,2024|1|loretta smith,Loretta Smith,1464141.21,1510,969.629940,100.895,0.07,48200.00,3599.745502,Smith,Smith,Loretta Smith,2024|1|loretta smith,matched_to_official_candidate
9,2024,city_council,Portland City Council,1,2024|1|steph routh,Steph Routh,201809.37,572,352.813584,8.720,0.20,31881.49,1821.093465,Routh,Routh,Steph Routh,2024|1|steph routh,matched_to_official_candidate
0,2024,city_council,Portland City Council,1,2024|1|candace avalos,Candace Avalos,177221.93,470,377.067936,78.260,0.08,21297.66,1364.949484,Avalos,Avalos,Candace Avalos,2024|1|candace avalos,matched_to_official_candidate
12,2024,city_council,Portland City Council,1,2024|1|timur ender,Timur Ender,157642.42,160,985.265125,184.870,0.24,13098.42,1835.055606,Ender,Ender,Timur Ender,2024|1|timur ender,matched_to_official_candidate
7,2024,city_council,Portland City Council,1,2024|1|orestar|durphy,Durphy,111237.32,391,284.494425,21.000,0.04,16300.00,1026.683289,Durphy,Durphy,NaN,NaN,unmatched_source_candidate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,2024,city_council,Portland City Council,4,2024|4|chloe mason,Chloe Mason,8723.41,33,264.345758,150.000,2.63,1995.00,356.051213,Mason,Mason,Chloe Mason,2024|4|chloe mason,matched_to_official_candidate
67,2024,city_council,Portland City Council,4,2024|4|mike dinapoli,Mike DiNapoli,7389.00,45,164.200000,15.050,0.70,1541.80,278.968760,DiNapoli,DiNapoli,Mike DiNapoli,2024|4|mike dinapoli,matched_to_official_candidate
72,2024,city_council,Portland City Council,4,2024|4|orestar|underdahl,Underdahl,1886.77,23,82.033478,40.570,0.18,390.00,100.408038,Underdahl,Underdahl,NaN,NaN,unmatched_source_candidate
66,2024,city_council,Portland City Council,4,2024|4|michael trimble,Michael Trimble,1349.86,22,61.357273,4.115,0.70,700.00,150.455618,trimble,trimble,Michael Trimble,2024|4|michael trimble,matched_to_official_candidate


## 7. Expenditure-size bins

We reuse the same numeric boundaries as the fundraising profiles only as a
simple scale representation. These are not substantive spending-purpose
categories.


In [7]:
BIN_LABELS = [
    "Micro",
    "Small",
    "Medium",
    "Large",
    "Mega",
]

BIN_EDGES = [
    -np.inf,
    25,
    100,
    250,
    1000,
    np.inf,
]

spending["spending_bin"] = pd.cut(
    spending["amount"],
    bins=BIN_EDGES,
    labels=BIN_LABELS,
    right=True,
    ordered=True,
)

display(
    spending["spending_bin"]
    .value_counts(sort=False)
    .to_frame("records")
)


,records
spending_bin,
Micro,4668
Small,3339
Medium,1793
Large,2278
Mega,1411


## 8. Long spending profile

In [8]:
profile_long = (
    spending
    .groupby(
        PROFILE_KEYS
        + [
            "spending_bin",
        ],
        observed=False,
        as_index=False,
        dropna=False,
    )
    .agg(
        spending_amount=("amount", "sum"),
        spending_count=("amount", "size"),
    )
)

profile_totals = (
    profile_long
    .groupby(
        PROFILE_KEYS,
        as_index=False,
        dropna=False,
    )
    .agg(
        profile_total_spending=("spending_amount", "sum"),
        profile_total_count=("spending_count", "sum"),
    )
)

profile_long = profile_long.merge(
    profile_totals,
    on=PROFILE_KEYS,
    how="left",
    validate="many_to_one",
)

profile_long["spending_amount_share"] = (
    profile_long["spending_amount"]
    / profile_long["profile_total_spending"]
)

profile_long["spending_count_share"] = (
    profile_long["spending_count"]
    / profile_long["profile_total_count"]
)

profile_long = profile_long.merge(
    identity_cols,
    on="profile_key",
    how="left",
    validate="many_to_one",
)

display(profile_long.head(15))


,year,contest_type,office,district,profile_key,candidate,spending_bin,spending_amount,spending_count,profile_total_spending,profile_total_count,spending_amount_share,spending_count_share,source_file_stem,source_candidate_name,canonical_candidate,candidate_key,linkage_status
0,2024,city_council,Portland City Council,1,2024|1|candace avalos,Candace Avalos,Micro,1353.50,150,177221.93,470,0.007637,0.319149,Avalos,Avalos,Candace Avalos,2024|1|candace avalos,matched_to_official_candidate
1,2024,city_council,Portland City Council,1,2024|1|candace avalos,Candace Avalos,Small,7027.16,116,177221.93,470,0.039652,0.246809,Avalos,Avalos,Candace Avalos,2024|1|candace avalos,matched_to_official_candidate
2,2024,city_council,Portland City Council,1,2024|1|candace avalos,Candace Avalos,Medium,12667.39,76,177221.93,470,0.071478,0.161702,Avalos,Avalos,Candace Avalos,2024|1|candace avalos,matched_to_official_candidate
3,2024,city_council,Portland City Council,1,2024|1|candace avalos,Candace Avalos,Large,51604.78,90,177221.93,470,0.291187,0.191489,Avalos,Avalos,Candace Avalos,2024|1|candace avalos,matched_to_official_candidate
4,2024,city_council,Portland City Council,1,2024|1|candace avalos,Candace Avalos,Mega,104569.10,38,177221.93,470,0.590046,0.080851,Avalos,Avalos,Candace Avalos,2024|1|candace avalos,matched_to_official_candidate
5,2024,city_council,Portland City Council,1,2024|1|cayle tern,Cayle Tern,Micro,195.10,25,13202.71,72,0.014777,0.347222,Tern,Tern,Cayle Tern,2024|1|cayle tern,matched_to_official_candidate
6,2024,city_council,Portland City Council,1,2024|1|cayle tern,Cayle Tern,Small,1104.67,17,13202.71,72,0.083670,0.236111,Tern,Tern,Cayle Tern,2024|1|cayle tern,matched_to_official_candidate
7,2024,city_council,Portland City Council,1,2024|1|cayle tern,Cayle Tern,Medium,1627.98,11,13202.71,72,0.123307,0.152778,Tern,Tern,Cayle Tern,2024|1|cayle tern,matched_to_official_candidate
8,2024,city_council,Portland City Council,1,2024|1|cayle tern,Cayle Tern,Large,10274.96,19,13202.71,72,0.778246,0.263889,Tern,Tern,Cayle Tern,2024|1|cayle tern,matched_to_official_candidate
9,2024,city_council,Portland City Council,1,2024|1|cayle tern,Cayle Tern,Mega,0.00,0,13202.71,72,0.000000,0.000000,Tern,Tern,Cayle Tern,2024|1|cayle tern,matched_to_official_candidate


## 9. Validate profile shares

In [9]:
validation = (
    profile_long
    .groupby(
        PROFILE_KEYS,
        as_index=False,
        dropna=False,
    )
    .agg(
        amount_share_sum=("spending_amount_share", "sum"),
        count_share_sum=("spending_count_share", "sum"),
    )
)

validation["amount_share_ok"] = np.isclose(
    validation["amount_share_sum"],
    1.0,
)

validation["count_share_ok"] = np.isclose(
    validation["count_share_sum"],
    1.0,
)

print(
    "All amount-share profiles valid:",
    validation["amount_share_ok"].all(),
)

print(
    "All count-share profiles valid:",
    validation["count_share_ok"].all(),
)

display(
    validation.loc[
        ~validation["amount_share_ok"]
        | ~validation["count_share_ok"]
    ]
)


All amount-share profiles valid: True
All count-share profiles valid: True


,year,contest_type,office,district,profile_key,candidate,amount_share_sum,count_share_sum,amount_share_ok,count_share_ok


## 10. Wide spending profile

In [10]:
metrics = [
    "spending_amount",
    "spending_amount_share",
    "spending_count",
    "spending_count_share",
]

wide_parts = []

for metric in metrics:
    part = (
        profile_long
        .pivot(
            index=PROFILE_KEYS,
            columns="spending_bin",
            values=metric,
        )
        .reindex(columns=BIN_LABELS)
        .fillna(0)
    )

    part.columns = [
        f"{metric}_{str(bin_name).lower()}"
        for bin_name in part.columns
    ]

    wide_parts.append(part)

profile_wide = pd.concat(
    wide_parts,
    axis=1,
).reset_index()

profile_wide = profile_wide.merge(
    candidate_summary,
    on=PROFILE_KEYS,
    how="left",
    validate="one_to_one",
)

print("Rows:", len(profile_wide))
print("Columns:", len(profile_wide.columns))
display(profile_wide.head())


Rows: 76
Columns: 38


,year,contest_type,office,district,profile_key,candidate,spending_amount_micro,spending_amount_small,spending_amount_medium,spending_amount_large,spending_amount_mega,spending_amount_share_micro,spending_amount_share_small,spending_amount_share_medium,spending_amount_share_large,spending_amount_share_mega,spending_count_micro,spending_count_small,spending_count_medium,spending_count_large,spending_count_mega,spending_count_share_micro,spending_count_share_small,spending_count_share_medium,spending_count_share_large,spending_count_share_mega,total_spending,expenditure_count,average_expenditure,median_expenditure,min_expenditure,max_expenditure,std_expenditure,source_file_stem,source_candidate_name,canonical_candidate,candidate_key,linkage_status
0,2024,city_council,Portland City Council,1,2024|1|candace avalos,Candace Avalos,1353.50,7027.16,12667.39,51604.78,104569.10,0.007637,0.039652,0.071478,0.291187,0.590046,150,116,76,90,38,0.319149,0.246809,0.161702,0.191489,0.080851,177221.93,470,377.067936,78.260,0.08,21297.66,1364.949484,Avalos,Avalos,Candace Avalos,2024|1|candace avalos,matched_to_official_candidate
1,2024,city_council,Portland City Council,1,2024|1|cayle tern,Cayle Tern,195.10,1104.67,1627.98,10274.96,0.00,0.014777,0.083670,0.123307,0.778246,0.000000,25,17,11,19,0,0.347222,0.236111,0.152778,0.263889,0.000000,13202.71,72,183.370972,80.000,0.18,868.60,243.060876,Tern,Tern,Cayle Tern,2024|1|cayle tern,matched_to_official_candidate
2,2024,city_council,Portland City Council,1,2024|1|david linn,David Linn,415.06,278.93,1540.00,2942.50,0.00,0.080182,0.053884,0.297499,0.568435,0.000000,45,6,9,6,0,0.681818,0.090909,0.136364,0.090909,0.000000,5176.49,66,78.431667,15.990,0.18,750.00,153.081951,Linn,Linn,David Linn,2024|1|david linn,matched_to_official_candidate
3,2024,city_council,Portland City Council,1,2024|1|deian salazar,Deian Salazar,404.30,829.16,1845.20,1055.76,1192.43,0.075899,0.155657,0.346396,0.198196,0.223853,45,15,12,3,1,0.592105,0.197368,0.157895,0.039474,0.013158,5326.85,76,70.090132,20.825,0.18,1192.43,154.362181,Salazar,Salazar,Deian Salazar,2024|1|deian salazar,matched_to_official_candidate
4,2024,city_council,Portland City Council,1,2024|1|joe furi,Joe Furi,26.25,110.00,283.75,330.00,0.00,0.035000,0.146667,0.378333,0.440000,0.000000,3,2,2,1,0,0.375000,0.250000,0.250000,0.125000,0.000000,750.00,8,93.750000,55.000,3.50,330.00,111.281577,Furi,Furi,Joe Furi,2024|1|joe furi,matched_to_official_candidate


## 11. Quick descriptive view

In [11]:
district_summary = (
    candidate_summary
    .groupby(
        [
            "year",
            "district",
        ],
        as_index=False,
    )
    .agg(
        candidate_source_rows=("profile_key", "nunique"),
        total_spending=("total_spending", "sum"),
        median_candidate_spending=("total_spending", "median"),
        min_candidate_spending=("total_spending", "min"),
        max_candidate_spending=("total_spending", "max"),
    )
)

display(district_summary)


,year,district,candidate_source_rows,total_spending,median_candidate_spending,min_candidate_spending,max_candidate_spending
0,2024,1,13,2296813.23,60028.060,750.00,1464141.21
1,2024,2,23,1573367.17,70605.480,1584.91,154825.11
2,2024,3,18,2307557.12,51146.815,672.19,1081644.06
3,2024,4,22,1676149.67,73964.110,325.00,291095.65


## 12. Export

In [12]:
summary_path = (
    OUTPUT_DIR
    / "orestar_candidate_spending_summary.csv"
)

long_path = (
    OUTPUT_DIR
    / "orestar_candidate_spending_profiles_long.csv"
)

wide_path = (
    OUTPUT_DIR
    / "orestar_candidate_spending_profiles_wide.csv"
)

candidate_summary.to_csv(
    summary_path,
    index=False,
)

profile_long.to_csv(
    long_path,
    index=False,
)

profile_wide.to_csv(
    wide_path,
    index=False,
)

print("SAVED", summary_path)
print("SAVED", long_path)
print("SAVED", wide_path)


SAVED /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/spending/2024/city_council/orestar_candidate_spending_summary.csv
SAVED /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/spending/2024/city_council/orestar_candidate_spending_profiles_long.csv
SAVED /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/spending/2024/city_council/orestar_candidate_spending_profiles_wide.csv
